In [1]:
import torch

In [2]:
model = torch.hub.load('facebookresearch/pytorchvideo',
                                       model="i3d_r50",
                                       pretrained=True)

Using cache found in C:\Users\ASUS/.cache\torch\hub\facebookresearch_pytorchvideo_main


In [3]:
totalChildren = 0
for idx, child in enumerate(model.children()):
    print('==================================')
    print('#'+str(idx)+': '+str(child))
    totalChildren = totalChildren + 1

    totalChildrenCurr = 0
    for idx, node in enumerate(child.children()):
        print('##'+str(idx)+': '+str(node))
        totalChildrenCurr = totalChildrenCurr + 1

    print('Total nodes in this child: '+str(totalChildrenCurr))
print('Total children in the model: '+str(totalChildren))


#0: ModuleList(
  (0): ResNetBasicStem(
    (conv): Conv3d(3, 64, kernel_size=(5, 7, 7), stride=(1, 2, 2), padding=(2, 3, 3), bias=False)
    (norm): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (activation): ReLU()
    (pool): MaxPool3d(kernel_size=(1, 3, 3), stride=(1, 2, 2), padding=[0, 1, 1], dilation=1, ceil_mode=False)
  )
  (1): ResStage(
    (res_blocks): ModuleList(
      (0): ResBlock(
        (branch1_conv): Conv3d(64, 256, kernel_size=(1, 1, 1), stride=(1, 1, 1), bias=False)
        (branch1_norm): BatchNorm3d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (branch2): BottleneckBlock(
          (conv_a): Conv3d(64, 64, kernel_size=(3, 1, 1), stride=(1, 1, 1), padding=(1, 0, 0), bias=False)
          (norm_a): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (act_a): ReLU()
          (conv_b): Conv3d(64, 64, kernel_size=(1, 3, 3), stride=(1, 1, 1), padding=(0, 1, 1), 

In [4]:
devModel = [child for child in model.children()]
devModel1 = [child for child in devModel[0].children()]
# print(devModel1)
devModel2 = torch.nn.Sequential(*(devModel1[:-1]))

In [5]:
print(len(devModel1))

7


In [6]:
head = devModel1[-1]
device = torch.device('cuda:0')
poolLayer = head.pool
poolLayer.to(device)

AvgPool3d(kernel_size=(4, 7, 7), stride=(1, 1, 1), padding=(0, 0, 0))

In [10]:
for item in model.modules():
    for blk in item.modules():
        print('====================')
        print(blk)

Net(
  (blocks): ModuleList(
    (0): ResNetBasicStem(
      (conv): Conv3d(3, 64, kernel_size=(5, 7, 7), stride=(1, 2, 2), padding=(2, 3, 3), bias=False)
      (norm): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (activation): ReLU()
      (pool): MaxPool3d(kernel_size=(1, 3, 3), stride=(1, 2, 2), padding=[0, 1, 1], dilation=1, ceil_mode=False)
    )
    (1): ResStage(
      (res_blocks): ModuleList(
        (0): ResBlock(
          (branch1_conv): Conv3d(64, 256, kernel_size=(1, 1, 1), stride=(1, 1, 1), bias=False)
          (branch1_norm): BatchNorm3d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (branch2): BottleneckBlock(
            (conv_a): Conv3d(64, 64, kernel_size=(3, 1, 1), stride=(1, 1, 1), padding=(1, 0, 0), bias=False)
            (norm_a): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (act_a): ReLU()
            (conv_b): Conv3d(64, 64, kernel_size=(1,

In [16]:
device = torch.device('cuda:0')
model = devModel2.to(device)
poolLayer = torch.nn.AdaptiveAvgPool3d(1).to(device)

input = torch.randn(size=(5,3,32,128,128)).to(device)
op = model(input)
op = poolLayer(op)
op = torch.squeeze(op,(2,3,4))
print(op.size())
torch.cuda.empty_cache()

torch.Size([5, 2048])


In [ ]:
torch.cuda.empty_cache()

: 